In [1]:
"""
Pipeline:
1. Load physiological data (ECG + EDA)
2. Extract features with subject-specific normalization
3. Predict current emotion (valence/arousal)
4. Get target emotion from user
5. Recommend music playlist from MuSe dataset (90k songs)
"""

'\nPipeline:\n1. Load physiological data \n2. Extract features, subject-specific normalization\n3. Predict current emotion (v/a)\n4. Get target emotion from user\n5. Recommend Spotify playlist using search API\n'

In [1]:
import os
import pandas as pd
import numpy as np
import neurokit2 as nk
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from typing import Optional, Dict
import warnings
warnings.filterwarnings('ignore')

In [2]:
# CHANGE THESE!!!!!!!!!!!!
PHYSIO_DIR = r'C:\Users\brand\Downloads\SRP Music Rec\Interpolated\Physiological'
ANNOT_DIR = r'C:\Users\brand\Downloads\SRP Music Rec\Interpolated\Annotated'
MUSIC_DATABASE_PATH = r'C:\Users\brand\Downloads\SRP Music Rec\muse_dataset.csv' 

In [3]:
class EmotionPredictor:
    #Subject specific normalization, NO windowing
    def __init__(self):
        self.model = None
        self.subject_stats = {}
        self.feature_names = [
            'HR_mean', 'HR_std', 'HRV_SDNN', 'HRV_RMSSD', 'HRV_pNN50', 'HRV_MeanNN',
            'EDA_mean', 'EDA_std', 'EDA_phasic', 'EDA_phasic_std', 'EDA_tonic', 
            'EDA_peaks', 'EDA_peak_rate'
        ]
    
    def extract_features(self, ecg_data, eda_data, sampling_rate=1000):
        features = {}
        
        try:
            # ECG/Heart features
            ecg_cleaned = nk.ecg_clean(ecg_data, sampling_rate=sampling_rate)
            signals, info = nk.ecg_process(ecg_cleaned, sampling_rate=sampling_rate)
            hrv = nk.hrv_time(signals, sampling_rate=sampling_rate, show=False)
            
            features['HR_mean'] = signals["ECG_Rate"].mean()
            features['HR_std'] = signals["ECG_Rate"].std()
            features['HRV_SDNN'] = hrv["HRV_SDNN"].values[0] if "HRV_SDNN" in hrv.columns else 0
            features['HRV_RMSSD'] = hrv["HRV_RMSSD"].values[0] if "HRV_RMSSD" in hrv.columns else 0
            features['HRV_pNN50'] = hrv["HRV_pNN50"].values[0] if "HRV_pNN50" in hrv.columns else 0
            features['HRV_MeanNN'] = hrv["HRV_MeanNN"].values[0] if "HRV_MeanNN" in hrv.columns else 0
            
            # EDA features
            eda_signals, eda_info = nk.eda_process(eda_data, sampling_rate=sampling_rate)
            features['EDA_mean'] = eda_data.mean()
            features['EDA_std'] = eda_data.std()
            features['EDA_phasic'] = eda_signals["EDA_Phasic"].mean()
            features['EDA_phasic_std'] = eda_signals["EDA_Phasic"].std()
            features['EDA_tonic'] = eda_signals["EDA_Tonic"].mean()
            features['EDA_peaks'] = len(eda_info["SCR_Peaks"])
            features['EDA_peak_rate'] = len(eda_info["SCR_Peaks"]) / (len(eda_data) / sampling_rate)
            
        except Exception as e:
            print(f"Feature extraction error: {e}")
            for key in self.feature_names:
                features[key] = 0
        
        return features
    
    def train_with_subject_normalization(self, physio_dir, annot_dir, num_subjects=30):
        print("Training model")
        all_subjects_data = {}
        
        for sub in range(1, num_subjects + 1):
            try:
                df_physio = pd.read_csv(os.path.join(physio_dir, f"sub_{sub}.csv"))
                df_annot = pd.read_csv(os.path.join(annot_dir, f"sub_{sub}.csv"))
                
                features = self.extract_features(
                    df_physio["ecg"].values,
                    df_physio["gsr"].values
                )
                
                valence = df_annot["valence"].mean()
                arousal = df_annot["arousal"].mean()
                
                all_subjects_data[sub] = {
                    'features': np.array(list(features.values())),
                    'valence': valence,
                    'arousal': arousal
                }
                
            except Exception as e:
                continue
        
        # Global std/normalization
        all_features = np.array([data['features'] for data in all_subjects_data.values()])
        global_std = np.std(all_features, axis=0) + 1e-8
        
        # Store subject-specific stats
        for sub_id, data in all_subjects_data.items():
            self.subject_stats[sub_id] = {
                'mean': data['features'],
                'std': global_std
            }
        
        # Subject specific normalization
        X_normalized = []
        y_normalized = []
        
        for sub_id, data in all_subjects_data.items():
            subject_mean = self.subject_stats[sub_id]['mean']
            subject_std = self.subject_stats[sub_id]['std']
            
            normalized = (data['features'] - subject_mean) / subject_std
            
            X_normalized.append(normalized)
            y_normalized.append([data['valence'], data['arousal']])
        
        X = np.array(X_normalized)
        y = np.array(y_normalized)
        
        # Train model
        self.model = MultiOutputRegressor(
            RandomForestRegressor(n_estimators=100, random_state=42, max_depth=5)
        )
        self.model.fit(X, y)
        
        # Calculate average baseline for new subjects
        self.avg_baseline = np.mean(all_features, axis=0)
        self.global_std = global_std
        
        print(f"Model trained on {len(X)} subjects with subject-specific normalization")
        print(f"(Best model: R² = -0.08)")
    
    def predict_emotion(self, ecg_data, eda_data, subject_id=None):
        
        if self.model is None:
            raise ValueError("MODEL NOT TRAINED")
        
        # Extract features
        features = self.extract_features(ecg_data, eda_data)
        feature_array = np.array(list(features.values()))
        
        if subject_id and subject_id in self.subject_stats:
            subject_mean = self.subject_stats[subject_id]['mean']
            subject_std = self.subject_stats[subject_id]['std']
        else:
            subject_mean = self.avg_baseline
            subject_std = self.global_std
        
        feature_normalized = (feature_array - subject_mean) / subject_std
        feature_normalized = feature_normalized.reshape(1, -1)
        
        # Predict
        prediction = self.model.predict(feature_normalized)
        
        valence = float(prediction[0][0])
        arousal = float(prediction[0][1])
        
        # Clip
        valence = np.clip(valence, -1, 1)
        arousal = np.clip(arousal, -1, 1)
        
        return valence, arousal


In [5]:
"""
class SpotifyMusicRecommender:
    #### SPOTIFY RECOMMENDER USING FREE API, (NOT USING SPOTIFY'S VALENCE/AROUSAL STUFF)
    
    def __init__(self, client_id, client_secret):
        auth_manager = SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
        self.sp = spotipy.Spotify(auth_manager=auth_manager)
        
        # Queries (so searching it up)
        self.emotion_queries = {
            'Q1_happy': ['happy pop', 'upbeat dance', 'feel good', 'party music', 'celebration'],
            'Q2_angry': ['hard rock', 'metal', 'aggressive', 'intense', 'powerful'],
            'Q3_sad': ['sad songs', 'heartbreak', 'emotional ballad', 'melancholy', 'somber'],
            'Q4_calm': ['relaxing', 'calm piano', 'ambient', 'meditation', 'peaceful'],
            'neutral': ['top hits', 'popular', 'mainstream']
        }
    
    def get_emotion_label(self, valence, arousal):
        if valence >= 0 and arousal >= 0:
            return 'Q1_happy', 'Happy/Excited'
        elif valence < 0 and arousal >= 0:
            return 'Q2_angry', 'Angry/Tense'
        elif valence < 0 and arousal < 0:
            return 'Q3_sad', 'Sad/Depressed'
        else:
            return 'Q4_calm', 'Calm/Relaxed'
    
    def search_by_emotion(self, quadrant, num_songs=5):
        queries = self.emotion_queries.get(quadrant, self.emotion_queries['neutral'])
        
        all_tracks = []
        seen_ids = set()
        
        for query in queries:
            if len(all_tracks) >= num_songs:
                break
            
            try:
                results = self.sp.search(q=query, type='track', limit=10, market='US')
                
                if results['tracks']['items']:
                    for track in results['tracks']['items']:
                        if track['id'] not in seen_ids:
                            all_tracks.append({
                                'title': track['name'],
                                'artist': ', '.join([a['name'] for a in track['artists']]),
                                'album': track['album']['name'],
                                'spotify_url': track['external_urls']['spotify'],
                                'search_term': query
                            })
                            seen_ids.add(track['id'])
                            
                            if len(all_tracks) >= num_songs:
                                break
            except Exception as e:
                print(f"Search error: {e}")
                continue
        
        return all_tracks[:num_songs]
    
    def recommend_playlist(self, current_valence, current_arousal,
                          target_valence, target_arousal, playlist_length=5):
        
        current_quad, current_desc = self.get_emotion_label(current_valence, current_arousal)
        target_quad, target_desc = self.get_emotion_label(target_valence, target_arousal)
        
        playlist = []
        
        if current_quad != target_quad and playlist_length >= 3:
            songs_per_section = max(1, playlist_length // 3)
            
            print(f"Searching Spotify for {current_desc} songs...")
            current_songs = self.search_by_emotion(current_quad, num_songs=songs_per_section)
            playlist.extend(current_songs)
            
            remaining = playlist_length - len(playlist)
            if remaining > 0:
                print("Searching for transition songs...")
                neutral_songs = self.search_by_emotion('neutral', num_songs=min(2, remaining))
                playlist.extend(neutral_songs)
            
            remaining = playlist_length - len(playlist)
            if remaining > 0:
                print("Searching for {target_desc} songs...")
                target_songs = self.search_by_emotion(target_quad, num_songs=remaining)
                playlist.extend(target_songs)
        else:
            print("Searching for {target_desc} songs...")
            playlist = self.search_by_emotion(target_quad, num_songs=playlist_length)
        
        return {
            'current_emotion': {'valence': current_valence, 'arousal': current_arousal,
                              'quadrant': current_quad, 'description': current_desc},
            'target_emotion': {'valence': target_valence, 'arousal': target_arousal,
                             'quadrant': target_quad, 'description': target_desc},
            'playlist': playlist
        }
"""

In [10]:
class MusicRecommender:
    #MUSE DATASET!!!!!!!!!! PREDOWNLOADED!!!!!!!!!!!!!

    #initialization/cleanup
    def __init__(self, database_path: str):
        print(f"Loading music database from: {database_path}")
        self.database = pd.read_csv(database_path)
        self._standardize_columns()
        print(f"Loaded {len(self.database)} songs")
        self._normalize_va_range()
    
    def _standardize_columns(self):
        rename_map = {
            'valence_tags': 'valence',
            'arousal_tags': 'arousal',
            'dominance_tags': 'dominance'
        }
        self.database = self.database.rename(columns=rename_map)
        required = ['valence', 'arousal']
        missing = [col for col in required if col not in self.database.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
            
    def _normalize_va_range(self):
        if self.database['valence'].min() >= 0 and self.database['valence'].max() <= 1:
            print("  Converting valence from [0,1] to [-1,1] range...")
            self.database['valence'] = self.database['valence'] * 2 - 1
        
        if self.database['arousal'].min() >= 0 and self.database['arousal'].max() <= 1:
            print("  Converting arousal from [0,1] to [-1,1] range...")
            self.database['arousal'] = self.database['arousal'] * 2 - 1

    #OTHERS
    
    def calculate_distance(self, song_valence: float, song_arousal: float,
                          target_valence: float, target_arousal: float) -> float:
        #pythag
        return np.sqrt((song_valence - target_valence)**2 + (song_arousal - target_arousal)**2)
    
    def search_by_emotion(self, target_valence: float, target_arousal: float,
                         genre: Optional[str] = None,
                         num_songs: int = 5) -> pd.DataFrame:
        # Filter by genre 
        if genre and 'genre' in self.database.columns:
            candidates = self.database[
                self.database['genre'].str.lower().str.contains(genre.lower(), na=False)
            ].copy()
            
            if candidates.empty:
                print(f"No songs found for genre '{genre}', using all songs...")
                candidates = self.database.copy()
        else:
            candidates = self.database.copy()
        
        candidates['distance'] = candidates.apply(lambda row: self.calculate_distance(row['valence'], row['arousal'], target_valence, target_arousal), axis=1)
        
        # Sort by distance (closest matches first)
        results = candidates.nsmallest(num_songs, 'distance')
        
        return results
    
    def get_emotion_label(self, valence: float, arousal: float) -> tuple:
        if valence >= 0 and arousal >= 0:
            return 'Q1', 'Happy/Excited'
        elif valence < 0 and arousal >= 0:
            return 'Q2', 'Angry/Tense'
        elif valence < 0 and arousal < 0:
            return 'Q3', 'Sad/Depressed'
        else:
            return 'Q4', 'Calm/Relaxed'
    
    def recommend_playlist(self, current_valence: float, current_arousal: float,
                          target_valence: float, target_arousal: float,
                          genre: Optional[str] = None,
                          playlist_length: int = 5,
                          gradual: bool = True) -> Dict:
        current_quad, current_desc = self.get_emotion_label(current_valence, current_arousal)
        target_quad, target_desc = self.get_emotion_label(target_valence, target_arousal)
        
        print(f"\nSearching for {playlist_length} songs...")
        print(f"From: {current_desc} (V={current_valence:.2f}, A={current_arousal:.2f})")
        print(f"To:   {target_desc} (V={target_valence:.2f}, A={target_arousal:.2f})")
        if genre:
            print(f"Genre: {genre}")
        
        all_songs = []
        used_ids = set()
        
        if gradual and playlist_length >= 3:
            valence_steps = np.linspace(current_valence, target_valence, playlist_length)
            arousal_steps = np.linspace(current_arousal, target_arousal, playlist_length)
            
            for i, (v, a) in enumerate(zip(valence_steps, arousal_steps)):
                songs = self.search_by_emotion(v, a, genre=genre, num_songs=10)
                
                for _, song in songs.iterrows():
                    song_id = song.get('id', f"{song.get('title', '')}_{song.get('artist', '')}")
                    
                    if song_id not in used_ids:
                        all_songs.append(song)
                        used_ids.add(song_id)
                        break
                
                if len(all_songs) >= playlist_length:
                    break
        else:
            songs = self.search_by_emotion(
                target_valence, target_arousal,
                genre=genre,
                num_songs=playlist_length
            )
            all_songs = [row for _, row in songs.iterrows()]
        
        return {
            'current_emotion': {
                'valence': current_valence,
                'arousal': current_arousal,
                'quadrant': current_quad,
                'description': current_desc
            },
            'target_emotion': {
                'valence': target_valence,
                'arousal': target_arousal,
                'quadrant': target_quad,
                'description': target_desc
            },
            'genre': genre,
            'transition_type': 'gradual' if gradual else 'direct',
            'playlist': all_songs[:playlist_length]
        }

In [5]:
#Joint class so everythings easier
class EmotionMusicSystem:
    
    def __init__(self, music_database_path: str):
        self.predictor = EmotionPredictor()
        self.recommender = MusicRecommender(music_database_path)
        self.trained = False
    
    def train_model(self, physio_dir, annot_dir):
        print("\n" + "-"*80)
        print("Training model")
        print("="*80)
        self.predictor.train_with_subject_normalization(physio_dir, annot_dir)
        self.trained = True
        print("Model trained\n")
    
    def process_and_recommend(self, ecg_data, eda_data, target_valence, target_arousal,
                             genre=None, subject_id=None):        
        if not self.trained:
            raise ValueError("MODEL NOT TRAINED YET")
        
        print("-"*80)
        print("Prediction + Reccomendation")
        print("-"*80)
        
        print("\n predicting")
        current_valence, current_arousal = self.predictor.predict_emotion(
            ecg_data, eda_data, subject_id
        )
        curr_quad, curr_desc = self.recommender.get_emotion_label(current_valence, current_arousal)
        
        print(f"\nPREDICTED CURRENT EMOTION:")
        print(f"{curr_desc} ({curr_quad})")
        print(f"Valence: {current_valence:.2f}")
        print(f"Arousal: {current_arousal:.2f}")
        
        # Display
        targ_quad, targ_desc = self.recommender.get_emotion_label(target_valence, target_arousal)
        print(f"\nTARGET EMOTION:")
        print(f"{targ_desc} ({targ_quad})")
        print(f"Valence: {target_valence:.2f}")
        print(f"Arousal: {target_arousal:.2f}")
        
        # Recommendation
        print(f"\nGenerating Personalized Playlist")
        result = self.recommender.recommend_playlist(
            current_valence, current_arousal,
            target_valence, target_arousal,
            genre=genre,
            playlist_length=5,
            gradual=True
        )
        
        # DONE
        self._display_playlist(result)
        
        return result

    def _display_playlist(self, recommendation):
        print(f"\n{'-'*80}")
        print("Reccommended playlist:")
        print("-"*80)
        
        if not recommendation['playlist']:
            print("\n????? NO PLAYLIST FOUND ??????")
            return
        
        for i, song in enumerate(recommendation['playlist'], 1):
            title = song.get('title', 'Unknown')
            artist = song.get('artist', 'Unknown Artist')
            valence = song.get('valence', 0)
            arousal = song.get('arousal', 0)
            distance = song.get('distance', 0)
            genre = song.get('genre', 'Unknown')
            
            print(f"\n{i}. '{title}' by {artist}")
            print(f"   Genre: {genre}")
            print(f"   Emotion: V={valence:.2f}, A={arousal:.2f}")
            print(f"   Match score: {distance:.3f} (lower = better)")
        
        print("\n" + "-"*80)
        print("Done")
        print("-"*80 + "\n")



In [11]:
########## MAIN ###############

if __name__ == "__main__":
    # MAIN

    #CHANGE THIS!!!!!!!!!!!
    if MUSIC_DATABASE_PATH == r'bah blah blah':
        print("NO MUSIC DATABASE FOUND")
    
    # Initialize system
    system = EmotionMusicSystem(MUSIC_DATABASE_PATH)
    system.train_model(PHYSIO_DIR, ANNOT_DIR)
    
    # USER INPUT DATA SHOUDL BE HERE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    subject_num = 1  #USER INPUT IS NUMBER 1!!!!
    df_physio = pd.read_csv(os.path.join(PHYSIO_DIR, f"sub_{subject_num}.csv"))
    ecg_data = df_physio["ecg"].values
    eda_data = df_physio["gsr"].values
    genre = 'pop'  #GENRE HERE!!!!!!!!!!!!!!!!!!!!!
    
    # ecg_data = np.array([])  
    # eda_data = np.array([])  
    # subject_num = None  
    
    # TARGET DATA HERE!!!!!!!!!!!!!!!!!!!!!!
    target_valence = -0.8   
    target_arousal = -0.7   

    result = system.process_and_recommend(ecg_data, eda_data, target_valence, target_arousal, genre=genre, subject_id=subject_num)

Loading music database from: C:\Users\brand\Downloads\SRP Music Rec\muse_dataset.csv
Loaded 90408 songs

--------------------------------------------------------------------------------
Training model
Training model
Model trained on 30 subjects with subject-specific normalization
(Best model: R² = -0.08)
Model trained

--------------------------------------------------------------------------------
Prediction + Reccomendation
--------------------------------------------------------------------------------

 predicting

PREDICTED CURRENT EMOTION:
Happy/Excited (Q1)
Valence: 1.00
Arousal: 1.00

TARGET EMOTION:
Sad/Depressed (Q3)
Valence: -0.80
Arousal: -0.70

Generating Personalized Playlist

Searching for 5 songs...
From: Happy/Excited (V=1.00, A=1.00)
To:   Sad/Depressed (V=-0.80, A=-0.70)
Genre: pop

--------------------------------------------------------------------------------
Reccommended playlist:
--------------------------------------------------------------------------------

1

In [7]:
import pandas as pd
df = pd.read_csv(MUSIC_DATABASE_PATH)
print(df.columns.tolist())
print(df.head(2))

['id', 'track', 'artist', 'valence_tags', 'arousal_tags', 'dominance_tags', 'mbid', 'spotify_id']
   id             track     artist  valence_tags  arousal_tags  \
0   0  'Till I Collapse     Eminem          4.55      5.273125   
1   1         St. Anger  Metallica          3.71      5.833000   

   dominance_tags                                  mbid  \
0        5.690625  cab93def-26c5-4fb0-bedd-26ec4c1619e1   
1        5.427250  727a2529-7ee8-4860-aef6-7959884895cb   

               spotify_id  
0  4xkOaSrkexMciUUogZKVTS  
1  3fOc9x06lKJBhz435mInlH  


In [ ]:
def dummy_data():
    import pandas as pd
    import numpy as np
    
    # dummy ECG/EDA data at 1000Hz, 10 seconds
    n = 10000
    t = np.linspace(0, 10, n)
    
    df = pd.DataFrame({
        "timestamp": t,
        "ecg": np.sin(2 * np.pi * 1.2 * t) + np.random.normal(0, 0.1, n),
        "eda": 2 + 0.5 * np.sin(2 * np.pi * 0.05 * t) + np.random.normal(0, 0.05, n)
    })
    
    df.to_csv("dummy_signal.csv", index=False)